# Preprocessing and tile extraction

Turns the whole-slide images into the tile dataset every model in this repository consumes.

**Contents** — removal of the 60 contaminated slides; construction of tissue masks by combining the provided annotation masks with intensity-based tissue detection; extraction of 256x256 tiles with a sliding window of stride 128 over the dilated mask; and the three filters that decide which tiles survive — tissue fraction below 20%, green staining artifacts detected in HSV space, and tumour mask coverage below 5%.

Each tile inherits the label of the slide it came from, which is what turns a slide-level classification problem into a tile-level one.

---


## Setup and paths

Originally executed on Google Colab with the dataset on Google Drive. Point
`WSI_DATA_DIR` at a directory holding the competition data (see `data/README.md`).


In [ ]:
# --- Paths ---------------------------------------------------------------
# Originally executed on Google Colab with the dataset on Google Drive.
# DATA_DIR must contain train_data/, test_data/ and train_labels.csv
# (see data/README.md).
import os

DATA_DIR = os.environ.get("WSI_DATA_DIR", "data")
os.makedirs("models", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)


## Outlier removal


In [29]:
import os
import glob

DATA_DIR = os.path.join(work_dir, "Dataset", "train_img_data")

# outlier IDs
shrek_ids = [5, 8, 22, 27, 36, 48, 62, 85, 95, 126, 129, 133, 136, 138, 148, 155, 159, 178, 179, 180, 187, 189, 193, 196, 251, 254, 263, 286, 313,319, 344, 346, 371, 376, 390, 393, 410, 415, 424, 443, 459, 498, 499, 521, 540, 544, 547, 558, 565, 572, 586, 602, 607, 609, 614, 620, 623, 646, 658, 673]

bad_ids = {str(i).zfill(4) for i in shrek_ids}

for filename in os.listdir(DATA_DIR):
    lower = filename.lower()


    if lower.startswith("img_") or lower.startswith("mask_"):

        id_part = lower.split("_")[1].split(".")[0]  # "0001" from img_0001.png

        if id_part in bad_ids:
            full_path = os.path.join(DATA_DIR, filename)
            try:
                os.remove(full_path)
                print("Deleted:", full_path)
            except Exception as e:
                print("Error:", full_path, e)

print("Done")

Done


## Train tiles generation


In [30]:
# import os
# import cv2
# import numpy as np
# import pandas as pd
# from tqdm import tqdm

# # ======================================
# # CONFIG
# # ======================================
# TRAIN_DIR = "Dataset/train_img_data/"
# TRAIN_CSV = "Dataset/train_labels.csv"
# OUT_TRAIN_RAW = "train_tiles_raw.npz"

# TILE_SIZE = 256
# STRIDE = 128
# DILATE_KERNEL = 15
# MIN_AREA = 700

# WHITE_THRESHOLD = 230
# TISSUE_FRAC_MIN = 0.20
# TUMOR_FRAC_MIN = 0.05

# GREEN_THRESHOLD_RATIO = 0.05
# LOWER_GREEN = np.array([35, 40, 40])
# UPPER_GREEN = np.array([85, 255, 255])

# CLASS_MAP = {
#     "Luminal A": 0,
#     "Luminal B": 1,
#     "HER2(+)": 2,
#     "Triple negative": 3
# }

# # ======================================
# # UTILS
# # ======================================
# def load_labels_dict(csv_path):
#     df = pd.read_csv(csv_path)
#     img_col = "image" if "image" in df.columns else df.columns[0]
#     lbl_col = "label" if "label" in df.columns else df.columns[1]
#     return {
#         row[img_col]: CLASS_MAP[row[lbl_col].strip()]
#         for _, row in df.iterrows()
#         if row[lbl_col].strip() in CLASS_MAP
#     }

# def is_background(tile, threshold=230, min_frac=0.2):
#     gray = cv2.cvtColor(tile, cv2.COLOR_RGB2GRAY)
#     tissue = np.sum(gray < threshold)
#     return (tissue / gray.size) < min_frac

# def has_green_artifact(tile):
#     hsv = cv2.cvtColor(tile, cv2.COLOR_RGB2HSV)
#     mask = cv2.inRange(hsv, LOWER_GREEN, UPPER_GREEN)
#     kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
#     mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
#     return (np.count_nonzero(mask) / mask.size) > GREEN_THRESHOLD_RATIO

# # ======================================
# # SINGLE IMAGE PROCESSING
# # ======================================
# def extract_tiles_from_image(img_path, mask_path):
#     img = cv2.imread(img_path)
#     mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
#     if img is None or mask is None:
#         return [], []

#     img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#     mask_bin = (mask > 0).astype(np.uint8)

#     kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (DILATE_KERNEL, DILATE_KERNEL))
#     mask_dil = cv2.dilate(mask_bin, kernel)

#     num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_dil, connectivity=8)

#     tiles, coords = [], []
#     H, W = mask.shape

#     for lab in range(1, num_labels):
#         if stats[lab, cv2.CC_STAT_AREA] < MIN_AREA:
#             continue

#         x0, y0, w, h = stats[lab, :4]
#         x1, y1 = min(x0 + w, W), min(y0 + h, H)

#         for y in range(y0, max(y0, y1 - TILE_SIZE + 1), STRIDE):
#             for x in range(x0, max(x0, x1 - TILE_SIZE + 1), STRIDE):
#                 if x + TILE_SIZE > W or y + TILE_SIZE > H:
#                     continue

#                 tile = img[y:y+TILE_SIZE, x:x+TILE_SIZE]
#                 mask_patch = mask_bin[y:y+TILE_SIZE, x:x+TILE_SIZE]

#                 if is_background(tile):
#                     continue
#                 if has_green_artifact(tile):
#                     continue
#                 if mask_patch.mean() < TUMOR_FRAC_MIN:
#                     continue

#                 tiles.append(tile)
#                 coords.append((x, y))

#     return tiles, coords

# # ======================================
# # DATASET LOOP
# # ======================================
# def build_train_dataset():
#     labels_dict = load_labels_dict(TRAIN_CSV)

#     all_tiles = []
#     all_labels = []
#     all_ids = []
#     all_coords = []

#     img_files = sorted(
#         f for f in os.listdir(TRAIN_DIR)
#         if f.startswith("img_") and f.endswith(".png") and f in labels_dict
#     )

#     for fname in tqdm(img_files):
#         img_path = os.path.join(TRAIN_DIR, fname)
#         mask_path = os.path.join(TRAIN_DIR, fname.replace("img_", "mask_"))

#         tiles, coords = extract_tiles_from_image(img_path, mask_path)
#         if not tiles:
#             continue

#         all_tiles.extend(tiles)
#         all_coords.extend(coords)
#         all_ids.extend([fname] * len(tiles))
#         all_labels.extend([labels_dict[fname]] * len(tiles))

#     np.savez_compressed(
#         OUT_TRAIN_RAW,
#         tiles=np.array(all_tiles, dtype=np.uint8),
#         labels=np.array(all_labels, dtype=np.int8),
#         slide_ids=np.array(all_ids),
#         coords=np.array(all_coords)
#     )

#     print(f"Saved {len(all_tiles)} train tiles")

# # ======================================
# # RUN
# # ======================================
# build_train_dataset()

In [36]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec

# ======================================
# CONFIG
# ======================================
TILE_SIZE = 256
STRIDE = 128
DILATE_KERNEL = 15
MIN_AREA = 700

WHITE_THRESHOLD = 230
TISSUE_FRAC_MIN = 0.20
TUMOR_FRAC_MIN = 0.05

GREEN_THRESHOLD_RATIO = 0.05
LOWER_GREEN = np.array([35, 40, 40])
UPPER_GREEN = np.array([85, 255, 255])

# ======================================
# UTILS
# ======================================
def is_background(tile, threshold=230, min_frac=0.2):
    """Check if tile is mostly background"""
    gray = cv2.cvtColor(tile, cv2.COLOR_RGB2GRAY)
    tissue = np.sum(gray < threshold)
    return (tissue / gray.size) < min_frac

def has_green_artifact(tile):
    """Check if tile contains green artifacts"""
    hsv = cv2.cvtColor(tile, cv2.COLOR_RGB2HSV)
    mask = cv2.inRange(hsv, LOWER_GREEN, UPPER_GREEN)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return (np.count_nonzero(mask) / mask.size) > GREEN_THRESHOLD_RATIO

# ======================================
# TILE EXTRACTION WITH VISUALIZATION
# ======================================
def extract_tiles_with_coords(img_path, mask_path):
    """Extract tiles and return their coordinates for visualization"""
    img = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if img is None or mask is None:
        raise ValueError(f"Could not load image or mask from:\n{img_path}\n{mask_path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask_bin = (mask > 0).astype(np.uint8)

    # Dilate mask to identify tissue regions
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (DILATE_KERNEL, DILATE_KERNEL))
    mask_dil = cv2.dilate(mask_bin, kernel)

    # Find connected components
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_dil, connectivity=8)

    valid_tiles = []
    all_boxes = []
    rejection_reasons = []
    H, W = mask.shape

    for lab in range(1, num_labels):
        if stats[lab, cv2.CC_STAT_AREA] < MIN_AREA:
            continue

        x0, y0, w, h = stats[lab, :4]
        x1, y1 = min(x0 + w, W), min(y0 + h, H)

        for y in range(y0, max(y0, y1 - TILE_SIZE + 1), STRIDE):
            for x in range(x0, max(x0, x1 - TILE_SIZE + 1), STRIDE):
                if x + TILE_SIZE > W or y + TILE_SIZE > H:
                    continue

                tile = img[y:y+TILE_SIZE, x:x+TILE_SIZE]
                mask_patch = mask_bin[y:y+TILE_SIZE, x:x+TILE_SIZE]

                # Check rejection criteria
                is_valid = True
                reason = "valid"

                if is_background(tile):
                    is_valid = False
                    reason = "background"
                elif has_green_artifact(tile):
                    is_valid = False
                    reason = "green_artifact"
                elif mask_patch.mean() < TUMOR_FRAC_MIN:
                    is_valid = False
                    reason = "low_tumor"

                all_boxes.append((x, y, is_valid, reason))
                if is_valid:
                    valid_tiles.append(tile)

    return img, mask_bin, all_boxes, valid_tiles

# ======================================
# VISUALIZATION
# ======================================
def visualize_tile_extraction(img_path, mask_path, output_path='tile_extraction_viz.png'):
    """Create visualization showing tile extraction process"""

    print(f"Processing image: {img_path}")
    img, mask, all_boxes, valid_tiles = extract_tiles_with_coords(img_path, mask_path)

    # Count rejection reasons
    valid_count = sum(1 for _, _, is_valid, _ in all_boxes if is_valid)
    bg_count = sum(1 for _, _, _, reason in all_boxes if reason == "background")
    green_count = sum(1 for _, _, _, reason in all_boxes if reason == "green_artifact")
    tumor_count = sum(1 for _, _, _, reason in all_boxes if reason == "insufficient_tissue")

    print(f"Total tiles: {len(all_boxes)}")
    print(f"Valid: {valid_count}")
    print(f"Rejected (background): {bg_count}")
    print(f"Rejected (green artifact): {green_count}")
    print(f"Rejected (low tumor): {tumor_count}")

    # Create figure with custom layout
    fig = plt.figure(figsize=(18, 7))
    gs = GridSpec(2, 6, figure=fig, hspace=0.3, wspace=0.3)

    # Main image with tile grid overlay
    ax_main = fig.add_subplot(gs[:, :3])
    ax_main.imshow(img)
    # ax_main.set_title('Raw Tile Extraction for MIL Approach', fontsize=14, fontweight='bold')
    ax_main.axis('off')

    # Draw all tile boxes with color coding
    for x, y, is_valid, reason in all_boxes:
        if is_valid:
            color = 'lime'
            alpha = 0.5
            linewidth = 2.0
        else:
            # Different colors for different rejection reasons
            if reason == "background":
                color = 'red'
                alpha = 0.15
                linewidth = 0.8
            elif reason == "green_artifact":
                color = 'orange'
                alpha = 0.3
                linewidth = 1.0
            else:  # low_tumor
                color = 'yellow'
                alpha = 0.2
                linewidth = 0.8

        rect = patches.Rectangle((x, y), TILE_SIZE, TILE_SIZE,
                                linewidth=linewidth, edgecolor=color,
                                facecolor='none', alpha=alpha)
        ax_main.add_patch(rect)

    # Add legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='lime', lw=3, label=f'Valid tiles ({valid_count})'),
        Line2D([0], [0], color='red', lw=2, alpha=0.5, label=f'Background ({bg_count})'),
        Line2D([0], [0], color='orange', lw=2, alpha=0.7, label=f'Green artifacts ({green_count})'),
        Line2D([0], [0], color='yellow', lw=2, alpha=0.5, label=f'Insufficient Tissue ({tumor_count})')
    ]
    ax_main.legend(handles=legend_elements, loc='upper right', fontsize=9)

    # Show sample extracted tiles (only valid ones)
    num_samples = min(6, len(valid_tiles))
    if num_samples > 0:
        sample_indices = np.linspace(0, len(valid_tiles)-1, num_samples, dtype=int)

        for i, idx in enumerate(sample_indices):
            row = i // 3
            col = i % 3 + 3
            ax = fig.add_subplot(gs[row, col])
            ax.imshow(valid_tiles[idx])
            ax.set_title(f'Valid Tile {idx+1}', fontsize=9)
            ax.axis('off')

    # plt.suptitle(f'Tile Extraction: {TILE_SIZE}×{TILE_SIZE} tiles with {STRIDE}px stride | '
    #              f'Image: {os.path.basename(img_path)}',
    #              fontsize=11, y=0.98)

    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"\nVisualization saved to: {output_path}")
    plt.close()

    return valid_count

# ======================================
# MAIN FUNCTION
# ======================================
if __name__ == "__main__":
    # MODIFY THESE PATHS TO YOUR IMAGE
    IMG_PATH = "Dataset/train_img_data/img_0023.png"  # <-- Change this
    MASK_PATH = "Dataset/train_mask_data/mask_0023.png"  # <-- Change this
    OUTPUT_PATH = "tile_extraction.png"  # <-- Change output path if needed

    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(OUTPUT_PATH) if os.path.dirname(OUTPUT_PATH) else ".", exist_ok=True)

    # Generate visualization
    try:
        visualize_tile_extraction(IMG_PATH, MASK_PATH, OUTPUT_PATH)
    except Exception as e:
        print(f"Error: {e}")
        print("\nPlease check:")
        print("1. IMG_PATH and MASK_PATH point to existing files")
        print("2. The paths are correct relative to your working directory")
        print("3. The files are valid PNG images")

Processing image: Dataset/train_img_data/img_0023.png
Total tiles: 9
  Valid: 9
  Rejected (background): 0
  Rejected (green artifact): 0
  Rejected (low tumor): 0

Visualization saved to: tile_extraction.png


In [27]:
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches

# def visualize_raw_tile_extraction(img_path, mask_path, max_tiles=200):
#     """
#     Visualise the raw tile extraction used for MIL:
#     the original image with the extracted tiles overlaid.
#     """
#     img = cv2.imread(img_path)
#     mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

#     img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#     mask_bin = (mask > 0).astype(np.uint8)

#     kernel = cv2.getStructuringElement(
#         cv2.MORPH_ELLIPSE, (DILATE_KERNEL, DILATE_KERNEL)
#     )
#     mask_dil = cv2.dilate(mask_bin, kernel)

#     num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
#         mask_dil, connectivity=8
#     )

#     H, W = mask.shape
#     tile_boxes = []

#     for lab in range(1, num_labels):
#         if stats[lab, cv2.CC_STAT_AREA] < MIN_AREA:
#             continue

#         x0, y0, w, h = stats[lab, :4]
#         x1, y1 = min(x0 + w, W), min(y0 + h, H)

#         for y in range(y0, max(y0, y1 - TILE_SIZE + 1), STRIDE):
#             for x in range(x0, max(x0, x1 - TILE_SIZE + 1), STRIDE):
#                 if x + TILE_SIZE > W or y + TILE_SIZE > H:
#                     continue

#                 tile = img[y:y+TILE_SIZE, x:x+TILE_SIZE]
#                 mask_patch = mask_bin[y:y+TILE_SIZE, x:x+TILE_SIZE]

#                 if is_background(tile):
#                     continue
#                 if has_green_artifact(tile):
#                     continue
#                 if mask_patch.mean() < TUMOR_FRAC_MIN:
#                     continue

#                 tile_boxes.append((x, y))
#                 if len(tile_boxes) >= max_tiles:
#                     break
#             if len(tile_boxes) >= max_tiles:
#                 break

#     # =============================
#     # PLOT
#     # =============================
#     fig, ax = plt.subplots(figsize=(10, 10))
#     ax.imshow(img)
#     ax.set_title("Raw tile extraction for MIL approach", fontsize=14)
#     ax.axis("off")

#     for (x, y) in tile_boxes:
#         rect = patches.Rectangle(
#             (x, y),
#             TILE_SIZE,
#             TILE_SIZE,
#             linewidth=1,
#             edgecolor="red",
#             facecolor="none",
#             alpha=0.6,
#         )
#         ax.add_patch(rect)

#     plt.tight_layout()
#     plt.show()

In [26]:
# img_example = "Dataset/train_img_data/img_0023.png"
# mask_example = "Dataset/train_mask_data/mask_0023.png"

# visualize_raw_tile_extraction(img_example, mask_example)

In [25]:
# import os
# import cv2
# import numpy as np
# import pandas as pd
# from tqdm import tqdm
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches

# # ======================================
# # CONFIG
# # ======================================
# TRAIN_DIR = "Dataset/train_img_data/"
# TRAIN_DIR_MSK = "Dataset/train_mask_data/"
# TRAIN_CSV = "Dataset/train_labels.csv"
# OUT_TRAIN_RAW = "train_tiles_raw.npz"

# TILE_SIZE = 256
# STRIDE = 128
# DILATE_KERNEL = 15
# MIN_AREA = 700

# WHITE_THRESHOLD = 230
# TISSUE_FRAC_MIN = 0.20
# TUMOR_FRAC_MIN = 0.05

# GREEN_THRESHOLD_RATIO = 0.05
# LOWER_GREEN = np.array([35, 40, 40])
# UPPER_GREEN = np.array([85, 255, 255])

# CLASS_MAP = {
#     "Luminal A": 0,
#     "Luminal B": 1,
#     "HER2(+)": 2,
#     "Triple negative": 3
# }

# # ======================================
# # UTILS
# # ======================================
# def load_labels_dict(csv_path):
#     df = pd.read_csv(csv_path)
#     img_col = "image" if "image" in df.columns else df.columns[0]
#     lbl_col = "label" if "label" in df.columns else df.columns[1]
#     return {
#         row[img_col]: CLASS_MAP[row[lbl_col].strip()]
#         for _, row in df.iterrows()
#         if row[lbl_col].strip() in CLASS_MAP
#     }

# def is_background(tile, threshold=WHITE_THRESHOLD, min_frac=TISSUE_FRAC_MIN):
#     gray = cv2.cvtColor(tile, cv2.COLOR_RGB2GRAY)
#     tissue = np.sum(gray < threshold)
#     return (tissue / gray.size) < min_frac

# def has_green_artifact(tile):
#     hsv = cv2.cvtColor(tile, cv2.COLOR_RGB2HSV)
#     mask = cv2.inRange(hsv, LOWER_GREEN, UPPER_GREEN)
#     kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
#     mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
#     return (np.count_nonzero(mask) / mask.size) > GREEN_THRESHOLD_RATIO

# # ======================================
# # TILE EXTRACTION
# # ======================================
# def extract_tiles_from_image(img_path, mask_path, mode='filtered', max_tiles=200):
#     """
#     Extract the tiles from one image.
#     mode='grid' -> mostra tutte le tiles (sliding window)
#     mode='filtered' -> applica i filtri originali MIL
#     """
#     img = cv2.imread(img_path)
#     mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
#     if img is None or mask is None:
#         return [], []

#     img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#     mask_bin = (mask > 0).astype(np.uint8)

#     kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (DILATE_KERNEL, DILATE_KERNEL))
#     mask_dil = cv2.dilate(mask_bin, kernel)

#     num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_dil, connectivity=8)

#     tile_boxes = []
#     H, W = mask.shape

#     for lab in range(1, num_labels):
#         if stats[lab, cv2.CC_STAT_AREA] < MIN_AREA:
#             continue

#         x0, y0, w, h = stats[lab, :4]
#         x1, y1 = min(x0 + w, W), min(y0 + h, H)

#         for y in range(y0, max(y0, y1 - TILE_SIZE + 1), STRIDE):
#             for x in range(x0, max(x0, x1 - TILE_SIZE + 1), STRIDE):
#                 if x + TILE_SIZE > W or y + TILE_SIZE > H:
#                     continue

#                 tile = img[y:y+TILE_SIZE, x:x+TILE_SIZE]
#                 mask_patch = mask_bin[y:y+TILE_SIZE, x:x+TILE_SIZE]

#                 if mode == 'filtered':
#                     if is_background(tile):
#                         continue
#                     if has_green_artifact(tile):
#                         continue
#                     if mask_patch.mean() < TUMOR_FRAC_MIN:
#                         continue

#                 tile_boxes.append((x, y))
#                 if len(tile_boxes) >= max_tiles:
#                     break
#             if len(tile_boxes) >= max_tiles:
#                 break

#     # =============================
#     # PLOT
#     # =============================
#     fig, ax = plt.subplots(figsize=(10, 10))
#     ax.imshow(img)
#     title = "Raw tile extraction (all tiles)" if mode=='grid' else "Filtered tiles (MIL)"
#     ax.set_title(title, fontsize=14)
#     ax.axis("off")

#     for (x, y) in tile_boxes:
#         rect = patches.Rectangle(
#             (x, y),
#             TILE_SIZE,
#             TILE_SIZE,
#             linewidth=1,
#             edgecolor="red",
#             facecolor="none",
#             alpha=0.6,
#         )
#         ax.add_patch(rect)

#     plt.tight_layout()
#     plt.show()

#     return tile_boxes

# # ======================================
# # DATASET BUILDING
# # ======================================
# def build_train_dataset():
#     labels_dict = load_labels_dict(TRAIN_CSV)

#     all_tiles = []
#     all_labels = []
#     all_ids = []
#     all_coords = []

#     img_files = sorted(
#         f for f in os.listdir(TRAIN_DIR)
#         if f.startswith("img_") and f.endswith(".png") and f in labels_dict
#     )

#     for fname in tqdm(img_files):
#         img_path = os.path.join(TRAIN_DIR, fname)
#         mask_path = os.path.join(TRAIN_DIR, fname.replace("img_", "mask_"))

#         tiles_coords = extract_tiles_from_image(img_path, mask_path, mode='filtered')
#         if not tiles_coords:
#             continue

#         # Extract the actual tiles that will be written to the dataset
#         img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
#         mask_bin = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) > 0

#         for (x, y) in tiles_coords:
#             tile = img[y:y+TILE_SIZE, x:x+TILE_SIZE]
#             all_tiles.append(tile)
#             all_coords.append((x, y))
#             all_ids.append(fname)
#             all_labels.append(labels_dict[fname])

#     np.savez_compressed(
#         OUT_TRAIN_RAW,
#         tiles=np.array(all_tiles, dtype=np.uint8),
#         labels=np.array(all_labels, dtype=np.int8),
#         slide_ids=np.array(all_ids),
#         coords=np.array(all_coords)
#     )

#     print(f"Saved {len(all_tiles)} train tiles")

# # ======================================
# # ESEMPIO DI VISUALIZZAZIONE
# # ======================================
# img_example = os.path.join(TRAIN_DIR, "img_0023.png")
# mask_example = os.path.join(TRAIN_DIR_MSK, "mask_0023.png")

# # Griglia completa
# extract_tiles_from_image(img_example, mask_example, mode='grid')

# # Filtered MIL tiles only
# extract_tiles_from_image(img_example, mask_example, mode='filtered')

## Test tiles generation


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# ======================================
# CONFIG
# ======================================
TEST_DIR = "data/test_data/"
OUT_TEST_RAW = "test_tiles_raw.npz"

TILE_SIZE = 256
STRIDE = 128
DILATE_KERNEL = 15
MIN_AREA = 700

WHITE_THRESHOLD = 230
TISSUE_FRAC_MIN = 0.20
TUMOR_FRAC_MIN = 0.05

# ======================================
# UTILS
# ======================================
def is_background(tile, threshold=230, min_frac=0.2):
    gray = cv2.cvtColor(tile, cv2.COLOR_RGB2GRAY)
    tissue = np.sum(gray < threshold)
    return (tissue / gray.size) < min_frac

# ======================================
# SINGLE IMAGE PROCESSING
# ======================================
def extract_tiles_from_image(img_path, mask_path):
    img = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None:
        return [], []

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask_bin = (mask > 0).astype(np.uint8)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (DILATE_KERNEL, DILATE_KERNEL))
    mask_dil = cv2.dilate(mask_bin, kernel)

    num_labels, _, stats, centroids = cv2.connectedComponentsWithStats(mask_dil, connectivity=8)

    tiles, coords = [], []
    H, W = mask.shape

    for lab in range(1, num_labels):
        if stats[lab, cv2.CC_STAT_AREA] < MIN_AREA:
            continue

        x0, y0, w, h = stats[lab, :4]
        x1, y1 = min(x0 + w, W), min(y0 + h, H)

        found_any = False

        for y in range(y0, max(y0, y1 - TILE_SIZE + 1), STRIDE):
            for x in range(x0, max(x0, x1 - TILE_SIZE + 1), STRIDE):
                if x + TILE_SIZE > W or y + TILE_SIZE > H:
                    continue

                tile = img[y:y+TILE_SIZE, x:x+TILE_SIZE]
                mask_patch = mask_bin[y:y+TILE_SIZE, x:x+TILE_SIZE]

                if is_background(tile):
                    continue
                if mask_patch.mean() < TUMOR_FRAC_MIN:
                    continue

                tiles.append(tile)
                coords.append((x, y))
                found_any = True

        # Fallback: center crop of the blob
        if not found_any:
            cx, cy = map(int, centroids[lab])
            half = TILE_SIZE // 2

            x_c = min(max(0, cx - half), max(0, W - TILE_SIZE))
            y_c = min(max(0, cy - half), max(0, H - TILE_SIZE))

            tile = img[y_c:y_c+TILE_SIZE, x_c:x_c+TILE_SIZE]
            if not is_background(tile):
                tiles.append(tile)
                coords.append((x_c, y_c))

    return tiles, coords

# ======================================
# DATASET LOOP
# ======================================
def build_test_dataset():
    img_files = sorted(
        f for f in os.listdir(TEST_DIR)
        if f.startswith("img_") and f.endswith(".png")
    )

    all_tiles = []
    all_ids = []
    all_coords = []
    all_labels = []

    for fname in tqdm(img_files):
        img_path = os.path.join(TEST_DIR, fname)
        mask_path = os.path.join(TEST_DIR, fname.replace("img_", "mask_"))
        if not os.path.exists(mask_path):
            continue

        tiles, coords = extract_tiles_from_image(img_path, mask_path)
        if not tiles:
            continue

        all_tiles.extend(tiles)
        all_coords.extend(coords)
        all_ids.extend([fname] * len(tiles))
        all_labels.extend([-1] * len(tiles))

    np.savez_compressed(
        OUT_TEST_RAW,
        tiles=np.array(all_tiles, dtype=np.uint8),
        slide_ids=np.array(all_ids),
        coords=np.array(all_coords),
        labels=np.array(all_labels, dtype=np.int8)
    )

    print(f"Saved {len(all_tiles)} test tiles from {len(np.unique(all_ids))} images")

# ======================================
# RUN
# ======================================
build_test_dataset()